In [1]:
!pip install sentence-transformers

!pip install faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 95.6 MB/s eta 0:00:00


In [2]:
import pandas as pd

import numpy as np

from sentence_transformers import SentenceTransformer

import faiss

import pickle

In [3]:
movies = pd.read_csv("processed_movies.csv")

movies.head()

,id,title,tags,vote_average,vote_count,popularity,release_date,original_language
0,862,Toy Story,"led by woody, andy' toy live happili in hi roo...",7.7,5415.0,21.946943,1995-10-30,en
1,8844,Jumanji,when sibl judi and peter discov an enchant boa...,6.9,2413.0,17.015539,1995-12-15,en
2,15602,Grumpier Old Men,a famili wed reignit the ancient feud between ...,6.5,92.0,11.712900,1995-12-22,en
3,31357,Waiting to Exhale,"cheat on, mistreat and step on, the women are ...",6.1,34.0,3.859495,1995-12-22,en
4,11862,Father of the Bride Part II,just when georg bank ha recov from hi daughter...,5.7,173.0,8.387519,1995-02-10,en


In [4]:
movies.shape

(45548, 8)

In [5]:
model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [8]:
embeddings = model.encode(

    movies["tags"].tolist(),

    batch_size=32,

    show_progress_bar=True,

    convert_to_numpy=True

)

Batches:   0%|          | 0/1424 [00:00<?, ?it/s]

In [9]:
embeddings.shape

(45548, 384)

In [10]:
faiss.normalize_L2(embeddings)

In [11]:
dimension = embeddings.shape[1]

index = faiss.IndexFlatIP(dimension)

index.add(embeddings)

In [12]:
movie_index = movies[
    movies["title"]=="Interstellar"
].index[0]

query = embeddings[movie_index].reshape(1,-1)

scores,indices = index.search(query,10)

In [13]:
for idx in indices[0]:

    print(

        movies.iloc[idx]["title"]

    )

Interstellar
Passengers
The Day the Earth Stood Still
The Fifth Element
Millennium
Royal Space Force - The Wings Of Honneamise
Science Fiction Volume One: The Osiris Child
Prometheus
Meet Dave
The Inhabited Island


In [14]:
np.save(

    "movie_embeddings.npy",

    embeddings

)

In [15]:
faiss.write_index(

    index,

    "faiss.index"

)

In [16]:
pickle.dump(

    movies,

    open(

        "movie_metadata.pkl",

        "wb"

    )

)